In [262]:
import pandas as pd
import glob
import os
import datetime
folder_path = "tefas"
csv_files = sorted(glob.glob(os.path.join(folder_path, "*.csv")))
dfs = {os.path.basename(f).split('.')[0]: pd.read_csv(f) for f in csv_files}
funds = dfs[list(dfs.keys())[-1]].iloc[:,1:3]
myfunds = funds[~funds["Fon Adı"].str.contains("DÖVİZ|YABANCI|ÖZEL SEKTÖR|EUROBOND|DOLAR|AVRO", case=True, na=False)]
fundfilter = set(myfunds["Fon Kodu"])
all_funds_df = pd.DataFrame({"Fon Kodu": list(fundfilter)})
final_dfs = [] 
columns_to_float = ['Fiyat','Tedavüldeki Pay Sayısı','Fon Toplam Değer']

for date, df in dfs.items():
    newdf = df[df["Fon Kodu"].isin(fundfilter)]
    newdf = all_funds_df.merge(newdf, on="Fon Kodu", how="left").fillna(0)
    newdf['Tarih']=pd.to_datetime(date, format='%m%d%Y')

    newdf[columns_to_float] = (
    newdf[columns_to_float]
    .replace(r"\.", "", regex=True)
    .replace(",", ".", regex=True)
    .astype(float)
)
    
    final_dfs.append(newdf)
final_df = pd.concat(final_dfs, ignore_index=True)

fund_dfs = {}
for fund, df_fund in final_df.groupby("Fon Kodu"):
    df_fund = df_fund.sort_values(by="Tarih").reset_index().drop(columns='index')
    df_fund['Pay Değişimi']=df_fund['Tedavüldeki Pay Sayısı'].diff().fillna(0)
    df_fund['Fon Giriş Çıkış']=df_fund['Pay Değişimi']*df_fund['Fiyat']
    fund_dfs[fund] = df_fund

In [263]:
pd.options.display.float_format = '{:,.2f}'.format 
combined_df = pd.concat(fund_dfs.values())  # Extract DataFrames from dictionary

# Sum the target column (e.g., "VALUE") for each date
summed_df = combined_df.groupby("Tarih", as_index=False)["Fon Giriş Çıkış"].sum()

summed_df.iloc[1:]


,Tarih,Fon Giriş Çıkış
1,2025-01-03,"1,955,623,101.03"
2,2025-01-06,"1,293,292,530.33"
3,2025-01-07,"498,214,408.91"
4,2025-01-08,"1,225,592,187.86"
5,2025-01-09,"1,608,722,094.81"
6,2025-01-10,"762,368,573.73"
7,2025-01-13,"1,311,174,522.02"
8,2025-01-14,"783,908,754.21"
9,2025-01-15,"1,987,813,021.17"
10,2025-01-16,"3,981,483,684.91"
